In [ ]:
import random
import time
import numpy as np
from collections import defaultdict

In [ ]:
folder = '/home/user/Desktop/ml/gpt_dt/datasets/instruction_fine_tuning'
split = 'train'

In [ ]:
# PACKETIZE DA DATA

# Tell the script how many of each block type are in the file
num_512_blocks = 229_528  # Change these to your actual numbers
num_256_blocks = 268_024
num_128_blocks = 337_519

pad_token_id = 30000
target_block_size = 1024

input_file =f'{folder}/{split}.bin'
output_file = f'{folder}/packed_{split}_{target_block_size}.bin'

def process_efficiently():
    start_time = time.time()
    
    # 1. Map input (Uses ~0 RAM)
    all_data = np.memmap(input_file, dtype=np.uint16, mode='r')
    
    # 2. SCAN: Get (offset, length) for every sequence
    # We use a structured array for metadata to keep RAM usage low
    metadata = np.zeros(num_512_blocks + num_256_blocks + num_128_blocks, 
                        dtype=[('offset', 'u8'), ('length', 'u2')])
    
    print("Scanning file for sequence lengths...")
    idx = 0
    ptr = 0
    for count, size in [(num_512_blocks, 512), (num_256_blocks, 256), (num_128_blocks, 128)]:
        for _ in range(count):
            chunk = all_data[ptr : ptr + size]
            # Find the first pad token to determine real length
            pad_indices = np.where(chunk == pad_token_id)[0]
            length = pad_indices[0] if len(pad_indices) > 0 else size
            
            metadata[idx] = (ptr, length)
            ptr += size
            idx += 1

    # 3. SORT: Biggest sequences first (The "D" in FFD)
    print("Sorting metadata by length...")
    metadata.sort(order='length')
    metadata = metadata[::-1] # Reverse for descending

    # 4. PACK: First Fit logic with Capacity Lookup
    print(f"Packing {len(metadata)} items into {target_block_size} bins...")
    
    # bin_contents will store indices of metadata for each 1024-block
    bins = []
    bin_spaces = [] # To track remaining space in each bin
    space_buckets = defaultdict(list)

    for ptr, length in metadata:
        found_bin_idx = -1
        
        # Look for a bin that has AT LEAST 'length' space left
        # We check from 'length' up to 1024
        for space_available in range(length, target_block_size + 1):
            if space_buckets[space_available]:
                found_bin_idx = space_buckets[space_available].pop()
                break
        
        if found_bin_idx != -1:
            # Found a fit! Put it in the bin
            bins[found_bin_idx].append((ptr, length))
            bin_spaces[found_bin_idx] += length
            
            # Re-calculate remaining space and put back in a new bucket
            new_remaining = target_block_size - bin_spaces[found_bin_idx]
            if new_remaining > 0:
                space_buckets[new_remaining].append(found_bin_idx)
        else:
            # No existing bin has enough space, create a new one
            new_idx = len(bins)
            bins.append([(ptr, length)])
            bin_spaces.append(length)
            
            # Calculate remaining space and put in a bucket
            remaining = target_block_size - length
            if remaining > 0:
                space_buckets[remaining].append(new_idx)

    # 5. WRITE: Transfer tokens directly to output memmap
    print(f"Writing {len(bins)} packed blocks to disk...")
    out_data = np.memmap(output_file, dtype=np.uint16, mode='w+', shape=(len(bins) * target_block_size + 1,))
    
    random.shuffle(bins)

    out_ptr = 0
    for b in bins:
        current_fill = 0
        for ptr, length in b:
            p, l = int(ptr), int(length)
            out_data[out_ptr : out_ptr + l] = all_data[p : p + l]
            out_ptr += l
            current_fill += l
        
        # Fill remaining gap with padding
        gap = target_block_size - current_fill
        if gap > 0:
            out_data[out_ptr : out_ptr + gap] = pad_token_id
            out_ptr += gap
    out_data[out_ptr] = pad_token_id # append one last pad to make it eASier for the dataloader which grabs batch+1 for the y

    out_data.flush()
    print(f"Finished in {time.time() - start_time:.2f} seconds.")

process_efficiently()

In [ ]:
# append one bin file to the end of another
folder = '/home/user/Desktop/Python_Scripts/gpt_course/gpt_dt/pretraining'
split = 'train'
def append_bin_files(source_path, destination_path):
    with open(source_path, 'rb') as f_src:
        with open(destination_path, 'ab') as f_dst:
            # Move in chunks to avoid RAM spikes
            while True:
                chunk = f_src.read(1024 * 1024) # 1MB chunks
                if not chunk:
                    break
                f_dst.write(chunk)

append_bin_files(f'{folder}/owt_{split}_data.bin', f'{folder}/all_{split}.bin')
append_bin_files(f'{folder}/owm_{split}_data.bin', f'{folder}/all_{split}.bin')
print("Done!")

In [ ]:
# appending one padding token to the end of the packed file
file_path = f'{folder}/packed_{split}_1024.bin'
pad_token_id = 30000
pad_bytes = np.array([pad_token_id], dtype=np.uint16).tobytes()

# 2. Open in 'ab' (Append Binary) mode
# This does NOT load the file into RAM
print(f"Appending pad token {pad_token_id} to {file_path}...")

try:
    with open(file_path, 'ab') as f:
        f.write(pad_bytes)
    print("Done! One token added to the end.")
except Exception as e:
    print(f"Failed to append: {e}")